# Detection Debugging
Load one frame from a clip, adjust YOLO confidence and image size, then inspect player and ball boxes.

In [ ]:
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = next(p for p in Path.cwd().resolve().parents if (p / 'src').exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_configs, resolve_project_path
from src.detection import detect_frame, load_model
from src.utils.drawing import draw_detections
from src.video_io import load_video_metadata, read_frame

In [ ]:
configs = load_configs()
camera_id = 'cam_01'
camera_config = configs['cameras'].get('cameras', configs['cameras']).get(camera_id, {})
VIDEO_PATH = resolve_project_path(camera_config.get('video_path', 'data/test.mp4'))
FRAME_INDEX = 0  # Change this to inspect another moment

metadata = load_video_metadata(VIDEO_PATH)
frame = read_frame(VIDEO_PATH, FRAME_INDEX)
print(metadata)

In [ ]:
detection_config = configs['detection']
model = load_model(detection_config.get('model_path', 'yolov8n.pt'), detection_config.get('device'))
PLAYER_CLASS_ID = int(detection_config.get('player_class_id', 0))
BALL_CLASS_ID = int(detection_config.get('ball_class_id', 32))
PLAYER_CONF = float(detection_config.get('player_confidence', 0.25))
BALL_CONF = float(detection_config.get('ball_confidence', 0.10))

def preview(confidence=0.25, imgsz=960):
    detections = detect_frame(
        model, frame,
        player_conf=confidence, ball_conf=BALL_CONF,
        player_class_id=PLAYER_CLASS_ID, ball_class_id=BALL_CLASS_ID,
        imgsz=imgsz, device=detection_config.get('device'),
    )
    rendered = draw_detections(frame, detections)
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()
    print(f'{len(detections)} detections at frame {FRAME_INDEX}')

widgets.interact(
    preview,
    confidence=widgets.FloatSlider(value=PLAYER_CONF, min=0.05, max=0.80, step=0.05, description='Player conf'),
    imgsz=widgets.SelectionSlider(options=[640, 960, 1280], value=960, description='Image size'),
);